# Pareidolia ML - Memoria del Proyecto

Este notebook documenta el resumen del proyecto completo de Machine Learning para la detección de Pareidolia (ilusión óptica de caras en objetos).

## 1. Estructura del Proyecto

```
Pareidolia_ML/
├── src/
│   ├── utils/                    # Módulos reutilizables
│   │   ├── constants.py          # Configuración y constantes
│   │   ├── data_loader.py        # Carga y procesamiento de datos
│   │   ├── model_builder.py      # Construcción de modelos
│   │   ├── training.py           # Funciones de entrenamiento
│   │   ├── evaluation.py         # Evaluación y métricas
│   │   └── prediction.py         # Predicciones
│   ├── data/                     # Datasets
│   │   ├── data.npz              # Datos normalizados
│   │   └── data_gray.npz         # Datos en escala de grises
│   ├── notebooks/                # Notebooks ejecutables
│   │   ├── 01_data_preparation.ipynb
│   │   ├── 02_model_comparison.ipynb
│   │   ├── 03_fine_tuning.ipynb
│   │   └── 04_predictions.ipynb
│   ├── model/                    # Modelos entrenados
│   │   ├── production/           # Modelo de producción final
│   │   └── *.keras               # Otros modelos experimentales
│   └── memoria.ipynb             # Este archivo - resumen del proyecto
├── resources/
│   └── img/                      # Imágenes y recursos visuales
├── data/                         # Datos originales (train/test)
├── model/                        # Modelos originales
├── notebooks/                    # Notebooks originales
└── README.md
```

## 2. Descripción del Proyecto

**Objetivo:** Construir un clasificador de imágenes capaz de detectar si una imagen contiene una pareidolia (ilusión óptica de una cara).

**Clases:**
- `cara`: Imágenes con pareidolia (cara detectada)
- `sin-cara`: Imágenes sin pareidolia (sin cara detectada)

**Enfoque:** Transfer Learning con arquitecturas preentrenadas (EfficientNetB0, ResNet50, Xception)

## 3. Paso 1: Preparación de Datos

**Archivo:** `src/notebooks/01_data_preparation.ipynb`

### Actividades:
1. **Carga de imágenes:** Lectura desde `data/train/` y `data/test/`
2. **Redimensionamiento:** A 224×224 píxeles
3. **Normalización:** Valores entre 0 y 1 (división entre 255)
4. **Barajado:** Mezcla aleatoria de datos de entrenamiento
5. **Guardado:** Serialización en `.npz` para carga rápida

### Resultados:
- Archivo: `src/data/data.npz`
- Distribución de clases verificada y equilibrada

## 4. Paso 2: Comparación de Modelos

**Archivo:** `src/notebooks/02_model_comparison.ipynb`

### Modelos Probados:
1. **EfficientNetB0** - Arquitectura eficiente, entrada 224×224
2. **ResNet50** - Arquitectura robusta, entrada 224×224
3. **Xception** - Especializado en texturas, entrada 299×299

### Arquitectura Común:
```
Backbone (congelado) → GlobalAveragePooling2D → 
BatchNormalization → Dense(128, relu) → Dropout(0.4) → 
Dense(1, sigmoid)
```

### Parámetros de Entrenamiento:
- Optimizer: Adam (lr=1e-4)
- Loss: Binary Crossentropy
- Metrics: Accuracy
- Callbacks: Early Stopping, ReduceLROnPlateau

### Resultado:
**✓ Mejor modelo:** Xception con AUC-ROC más alto

## 5. Paso 3: Fine-tuning

**Archivo:** `src/notebooks/03_fine_tuning.ipynb`

### Proceso:
1. Cargar modelo Xception entrenado
2. Descongelar últimas 25 capas del backbone
3. Recompilar con learning rate bajo (1e-5)
4. Entrenar 20 épocas adicionales con batch size menor (16)
5. Guardar modelo mejorado

### Mejoras:
- Adaptación más fina a características específicas del dataset
- Mejor rendimiento en validación
- Modelo guardado como `Xception_finetuned.keras`

## 6. Paso 4: Predicciones y Validación

**Archivo:** `src/notebooks/04_predictions.ipynb`

### Funcionalidades:
- Predicción en imágenes individuales
- Predicciones en lote
- Visualización de resultados
- Evaluación en dataset de test

### Métricas Reportadas:
- Accuracy total
- Accuracy por clase
- Matriz de confusión
- Curva ROC

## 7. Módulos Principales (`src/utils/`)

### 7.1 constants.py
- Configuración centralizada (dimensiones, rutas, hiperparámetros)

### 7.2 data_loader.py
- `read_data()`: Carga imágenes desde carpeta
- `load_and_prepare_data()`: Pipeline completo de carga
- `save_data_npz()`: Serialización a .npz
- `load_data_npz()`: Deserialización desde .npz

### 7.3 model_builder.py
- `build_model()`: Constructor genérico de modelos
- `build_efficient_net_b0()`, `build_resnet50()`, `build_xception()`
- `unfreeze_backbone_layers()`: Para fine-tuning

### 7.4 training.py
- `train_model()`: Entrenamiento básico
- `train_model_custom()`: Con validation set personalizado
- `train_with_augmentation()`: Con data augmentation

### 7.5 evaluation.py
- `evaluate_model()`: Evaluación completa
- `plot_confusion_matrix()`, `plot_learning_curves()`, `plot_roc_curve()`
- `find_optimal_threshold()`: Búsqueda de threshold

### 7.6 prediction.py
- `predict_single_image()`: Predicción en una imagen
- `batch_predict()`: Predicciones en lote
- `visualize_predictions()`: Grilla de predicciones

## 8. Modelo Final de Producción

**Ubicación:** `src/model/production/best_model.keras`

### Especificaciones:
- **Arquitectura:** Xception con fine-tuning
- **Entrada:** 299×299×3
- **Salida:** Probabilidad binaria (0-1)
- **Threshold:** 0.5
- **Performance:** [Insertar AUC-ROC, Accuracy, F1]

### Uso:
```python
from src.utils import predict_and_visualize
import tensorflow as tf

model = tf.keras.models.load_model('src/model/production/best_model.keras')
result = predict_and_visualize(model, 'ruta/imagen.jpg')
```

## 9. Cómo Usar los Notebooks

### Secuencia Recomendada:
1. **01_data_preparation.ipynb** → Prepara los datos
2. **02_model_comparison.ipynb** → Compara modelos
3. **03_fine_tuning.ipynb** → Mejora el mejor modelo
4. **04_predictions.ipynb** → Valida en dataset de test

### Requisitos:
- Python 3.8+
- TensorFlow >= 2.10
- Numpy, Pandas, Matplotlib, Scikit-learn

### Instalación:
```bash
pip install -r requirements.txt
```

## 10. Conclusiones y Próximos Pasos

### Hallazgos:
- Xception superó a otros backbones en este dataset
- Fine-tuning mejoro significativamente el rendimiento
- El modelo está listo para producción

### Mejoras Futuras:
- Implementar data augmentation más agresivo
- Probar ensemble de modelos
- Ajuste automático de threshold según caso de uso
- Despliegue como API REST
- Monitoreo de drift en producción

## 11. Referencias

- **Transfer Learning:** https://keras.io/guides/transfer_learning/
- **Xception:** Chollet, F. (2017). Xception: Deep Learning with Depthwise Separable Convolutions
- **EfficientNet:** Tan & Le (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks
- **ResNet:** He et al. (2015). Deep Residual Learning for Image Recognition